# Cadeia de suprimentos: o repositório de nome quase igual

OWASP Top 10 para LLMs 2025 — **LLM03 Supply Chain**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
Um laboratório precisa baixar os pesos de um modelo aberto. Existe o repositório oficial e um repositório impostor, com o nome quase idêntico (dois hifens em vez de um). **Nenhum carregamento real acontece neste notebook** — o "carregamento" é só uma simulação textual do que aconteceria.

In [2]:
print("oficial:", dados.REPO_OFICIAL)
print("golpe:  ", dados.REPO_GOLPE)


oficial: {'nome': 'aurora-labs/aurora-base-7b', 'hash': 'sha256:3f9a1c...oficial-sintetico'}
golpe:   {'nome': 'aurora-labs/aurora--base-7b', 'hash': 'sha256:9c1ee0...nao-verificado'}


In [3]:
def carregar_pesos_simulado(repo, verificar=False):
    """NÃO carrega nada de verdade (nenhum pickle.load, nenhuma rede). Só narra o que um
    carregador ingênuo faria com um arquivo de pesos em formato pickle malicioso."""
    if verificar and not defesas.repositorio_confiavel(repo["nome"], repo["hash"]):
        return "bloqueado: repositório fora da allowlist de hash/nome (nada foi carregado)"
    if repo is dados.REPO_GOLPE:
        return ("[SIMULADO] o arquivo .bin em formato pickle executaria código ao ser "
                "desserializado: leria a variável de ambiente AURORA_CLUSTER_TOKEN e a "
                "enviaria para um destino externo — nada disso foi executado de verdade aqui.")
    return "pesos carregados normalmente (repositório oficial)"


## Sem defesa

In [4]:
print(carregar_pesos_simulado(dados.REPO_GOLPE, verificar=False))


[SIMULADO] o arquivo .bin em formato pickle executaria código ao ser desserializado: leria a variável de ambiente AURORA_CLUSTER_TOKEN e a enviaria para um destino externo — nada disso foi executado de verdade aqui.


## Com defesa
Verifica nome **e** hash contra uma allowlist antes de qualquer carregamento.

In [5]:
print(carregar_pesos_simulado(dados.REPO_GOLPE, verificar=True))
print(carregar_pesos_simulado(dados.REPO_OFICIAL, verificar=True))


bloqueado: repositório fora da allowlist de hash/nome (nada foi carregado)
pesos carregados normalmente (repositório oficial)


## O que observar
- O nome do repositório impostor difere do oficial por um único caractere — fácil de não notar numa revisão rápida.
- Pesos, tokenizadores e adaptadores baixados de fora são código de terceiros: o risco não é "o modelo responder errado", é o processo de carregamento executar algo antes da primeira pergunta.
- A defesa (hash fixado, assinatura, allowlist) precisa rodar ANTES do carregamento; depois já é tarde.